In [ ]:
import pandas as pd

data = pd.read_csv("../sign_data.csv", header=None)

# Last column = label
labels = data.iloc[:, -1]

# Count samples per class
counts = labels.value_counts().sort_index()

print(counts)

42
A    1201
B    1377
C    1386
D    1493
E    1498
F    1656
G    1701
H    1251
I    1556
J    1319
K    1424
L    1462
M    1373
N    1322
O    1381
P    1187
Q    1521
R    1592
S    1751
T    1958
Name: count, dtype: int64


In [ ]:
!pip install torch torchvision torchaudio
!pip install torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 36.0 MB/s eta 0:00:00


In [ ]:
min_count = counts.min()

balanced_data = []
for label in counts.index:
    subset = data[data.iloc[:, -1] == label].sample(min_count)
    balanced_data.append(subset)

data = pd.concat(balanced_data).reset_index(drop=True)

In [ ]:
import torch
from torch_geometric.data import Data
from sklearn.preprocessing import LabelEncoder
import pickle

# Encode labels
le = LabelEncoder()
y = le.fit_transform(data.iloc[:, -1])

# Save encoder
with open("label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

X = data.iloc[:, :-1].values

# Hand edges (MediaPipe structure)
edges = [
    (0,1),(1,2),(2,3),(3,4),
    (0,5),(5,6),(6,7),(7,8),
    (0,9),(9,10),(10,11),(11,12),
    (0,13),(13,14),(14,15),(15,16),
    (0,17),(17,18),(18,19),(19,20)
]

edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

dataset = []

for i in range(len(X)):
    coords = X[i].reshape(21, 2)
    x = torch.tensor(coords, dtype=torch.float)

    graph = Data(x=x, edge_index=edge_index, y=torch.tensor(y[i]))
    dataset.append(graph)

In [ ]:
from sklearn.model_selection import train_test_split

train_data, temp_data = train_test_split(dataset, test_size=0.3, stratify=y)
val_data, test_data = train_test_split(temp_data, test_size=0.5)

print(len(train_data), len(val_data), len(test_data))

16618 3561 3561


In [ ]:
from torch_geometric.loader import DataLoader

train_loader = DataLoader(train_data, batch_size=120, shuffle=True)
val_loader = DataLoader(val_data, batch_size=120)
test_loader = DataLoader(test_data, batch_size=120)

In [ ]:
import torch.nn as nn
from torch_geometric.nn import GCNConv, global_mean_pool, BatchNorm

class GNNModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.conv1 = GCNConv(2, 64)
        self.bn1 = BatchNorm(64)

        self.conv2 = GCNConv(64, 128)
        self.bn2 = BatchNorm(128)

        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(128, num_classes)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = torch.relu(x)
        x = self.dropout(x)

        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = torch.relu(x)
        x = self.dropout(x)

        x = global_mean_pool(x, batch)

        return self.fc(x)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = GNNModel(num_classes=len(set(y))).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)
criterion = nn.CrossEntropyLoss()

def evaluate(loader):
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out = model(batch)

            pred = out.argmax(dim=1)
            correct += (pred == batch.y).sum().item()
            total += batch.y.size(0)

    return correct / total

In [ ]:
epochs = 60

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch in train_loader:
        batch = batch.to(device)

        optimizer.zero_grad()
        out = model(batch)
        loss = criterion(out, batch.y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    train_acc = evaluate(train_loader)
    val_acc = evaluate(val_loader)

    print(f"Epoch {epoch+1}")
    print(f"Loss: {total_loss:.4f}")
    print(f"Train Acc: {train_acc:.4f}")
    print(f"Val Acc: {val_acc:.4f}")
    print("-"*40)

Epoch 1
Loss: 52.0038
Train Acc: 0.9492
Val Acc: 0.9466
----------------------------------------
Epoch 2
Loss: 51.4367
Train Acc: 0.9581
Val Acc: 0.9551
----------------------------------------
Epoch 3
Loss: 51.2614
Train Acc: 0.9538
Val Acc: 0.9500
----------------------------------------
Epoch 4
Loss: 49.4920
Train Acc: 0.9615
Val Acc: 0.9598
----------------------------------------
Epoch 5
Loss: 50.5678
Train Acc: 0.9567
Val Acc: 0.9542
----------------------------------------
Epoch 6
Loss: 49.7343
Train Acc: 0.9621
Val Acc: 0.9579
----------------------------------------
Epoch 7
Loss: 48.5906
Train Acc: 0.9641
Val Acc: 0.9607
----------------------------------------
Epoch 8
Loss: 48.6892
Train Acc: 0.9620
Val Acc: 0.9596
----------------------------------------
Epoch 9
Loss: 48.9013
Train Acc: 0.9566
Val Acc: 0.9525
----------------------------------------
Epoch 10
Loss: 47.8733
Train Acc: 0.9587
Val Acc: 0.9562
----------------------------------------
Epoch 11
Loss: 46.7264
Train 

In [ ]:
test_acc = evaluate(test_loader)
print("Final Test Accuracy:", test_acc)

Final Test Accuracy: 0.9755686604886268


In [ ]:
torch.save(model.state_dict(), "gnn_model.pth")

In [ ]:
from google.colab import files
files.download("gnn_model.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
files.download("label_encoder.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>